# Figure 3g

In [ ]:
# Pls run all codes of this file to get Figure 4F

# Load packages

In [1]:
import scimap as sm
import scanpy as sc

import anndata as ad

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from scipy.stats import wilcoxon
import os

import numpy as np

Running SCIMAP  2.2.11


# Load data

In [6]:
adata = ad.read_h5ad('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/adata_interaction_radius45_imageid.h5ad')

# scimap spatial pscore

In [12]:
immune_celltypes = ['CD68.Macrophages', 'CD206.Macrophages', 'CD4.T.cells', 'CD8.T.cells', 'Treg', 'Other.Immune']
Tumor_celltypes = ['Tumor.GCLC+VIM+', 'Tumor.GCLC+VIM-', 'Tumor.GCLC-VIM+']

In [ ]:
def calculate_pscore_for_celltypes(adata, score_by, x_coordinate, y_coordinate, phenotype, method, radius):
    """
    Calculate pscore for each combination of immune celltype and tumor celltype.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix.
        
    Returns:
    --------
    adata : AnnData
        Updated annotated data matrix with pscore calculations.
    """
    for immune_type in immune_celltypes:
        for tumor_type in Tumor_celltypes:
            # Create a label for this combination
            # Replace special characters to avoid naming issues
            safe_immune_type = immune_type.replace('.', '_')
            safe_tumor_type = tumor_type.replace('.', '_').replace('+', 'pos').replace('-', 'neg')
            label = f"pscore_{safe_tumor_type}_{safe_immune_type}"
            
            # Calculate pscore
            adata = sm.tl.spatial_pscore(
                adata,
                proximity = [tumor_type, immune_type],
                score_by = score_by,
                x_coordinate = x_coordinate,
                y_coordinate = y_coordinate,
                phenotype = phenotype,
                method = method,
                radius = radius,
                label = label
            )
    
    return adata

In [ ]:
adata_pscore = calculate_pscore_for_celltypes(adata,
                                              score_by='core_imageid',
                                              x_coordinate='X_centroid', 
                                              y_coordinate='Y_centroid',
                                              phenotype='GCLC_VIM',
                                              method='radius',
                                              radius=45)

# Correct values

In [32]:
pscore_list = ['pscore_Tumor_GCLCposVIMpos_CD68_Macrophages', 'pscore_Tumor_GCLCposVIMneg_CD68_Macrophages', 'pscore_Tumor_GCLCnegVIMpos_CD68_Macrophages',
 'pscore_Tumor_GCLCposVIMpos_CD206_Macrophages', 'pscore_Tumor_GCLCposVIMneg_CD206_Macrophages', 'pscore_Tumor_GCLCnegVIMpos_CD206_Macrophages',
 'pscore_Tumor_GCLCposVIMpos_CD4_T_cells', 'pscore_Tumor_GCLCposVIMneg_CD4_T_cells', 'pscore_Tumor_GCLCnegVIMpos_CD4_T_cells',
 'pscore_Tumor_GCLCposVIMpos_CD8_T_cells', 'pscore_Tumor_GCLCposVIMneg_CD8_T_cells', 'pscore_Tumor_GCLCnegVIMpos_CD8_T_cells',
 'pscore_Tumor_GCLCposVIMpos_Treg', 'pscore_Tumor_GCLCposVIMneg_Treg', 'pscore_Tumor_GCLCnegVIMpos_Treg',
 'pscore_Tumor_GCLCposVIMpos_Other_Immune', 'pscore_Tumor_GCLCposVIMneg_Other_Immune', 'pscore_Tumor_GCLCnegVIMpos_Other_Immune']

In [33]:

def calculate_proximity_metrics(adata_pscore, pscore_label):
    """
    Calculate proximity volume and density metrics for a given pscore label.
    
    Parameters:
    -----------
    adata_pscore : AnnData
        Annotated data matrix with proximity scores.
    pscore_label : str
        The proximity score label to process.
    
    Returns:
    --------
    proximity_score : DataFrame
        DataFrame containing the proximity metrics.
    """
    # Extract tumor status and cell type from the pscore label for filtering
    parts = pscore_label.split('_')
    tumor_status = parts[2]  # GCLC VIM status
    tumor_status = tumor_status.replace("pos", "+").replace("neg", "-")
    cell_type = '_'.join(parts[3:])  # The rest is the cell type
    
    # Convert to the format used in GCLC_VIM
    tumor = f"Tumor.{tumor_status}"
    cell_type_converted = cell_type.replace('_', '.')
    
    # Get the whole dataset
    wh_d = adata_pscore.obs.copy()
    whole_data = wh_d[['core_imageid', pscore_label, 'GCLC_VIM']]

    # Calculate proximity volume
    p_v = whole_data.groupby(['core_imageid', pscore_label], observed=False).size().unstack().fillna(0)
    
    # Find the interaction column - it should be the one that's not 'other'
    interaction_cols = [col for col in p_v.columns if col != 'other']
    if not interaction_cols:
        print(f"No interaction column found for {pscore_label}")
        return pd.DataFrame()
    
    interaction_col = interaction_cols[0]
    
    p_v['All Cells'] = p_v[interaction_col] + p_v.get("other", 0)
    p_v['Proximity Volume'] = p_v[interaction_col] / p_v['All Cells']
    p_v = p_v.fillna(0)  # replace NA
    p_v = p_v.replace([np.inf, -np.inf], 0)
    p_v = p_v.drop(columns='other', errors='ignore')
    
    # Filter for relevant cell types for proximity density
    w_d = whole_data[whole_data['GCLC_VIM'].isin([tumor, cell_type_converted])]
    p_d = w_d.groupby(['core_imageid', pscore_label], observed=False).size().unstack().fillna(0)
    
    # Find the interaction column in p_d
    interaction_cols_d = [col for col in p_d.columns if col != 'other']
    if not interaction_cols_d:
        print(f"No interaction column found in filtered data for {pscore_label}")
        return pd.DataFrame()
    
    interaction_col_d = interaction_cols_d[0]
    
    p_d['Celltype of interest'] = p_d[interaction_col_d] + p_d.get("other", 0)
    p_d['Proximity Density'] = p_d[interaction_col_d] / p_d['Celltype of interest']
    p_d = p_d.fillna(0)  # replace NA
    p_d = p_d.replace([np.inf, -np.inf], 0)
    p_d = p_d.drop(columns=[col for col in p_d.columns if col not in ['Celltype of interest', 'Proximity Density']])
    
    # Merge the results
    proximity_score = pd.merge(p_v, p_d, left_index=True, right_index=True)
    
    return proximity_score

# Process each pscore
for pscore in pscore_list:
    try:
        # Calculate proximity metrics
        proximity_score = calculate_proximity_metrics(adata_pscore, pscore)
        
        if not proximity_score.empty:
            # Store in adata_pscore.uns
            adata_pscore.uns[pscore] = proximity_score
    
    except Exception as e:
        print(f"Error processing {pscore}: {e}")
        


# Proximity Density dataframe

In [37]:

result_df = pd.DataFrame()

# Process each pscore
for pscore in pscore_list:
    # Extract the data from adata_pscore.uns
    if pscore in adata_pscore.uns:
        pscore_data = adata_pscore.uns[pscore]
        
        # Ensure 'core_imageid' is a column, not an index
        pscore_data = pscore_data.reset_index()
        # Extract core_imageid and Proximity Density
        temp_df = pscore_data[['core_imageid', 'Proximity Density']].copy()
        
        # Convert column name format from pscore_Tumor_GCLCposVIMpos_CD68_Macrophages 
        # to Tumor.GCLC+VIM+_CD68.Macrophages
        column_name = pscore.replace('pscore_', '')
        column_name = column_name.replace('Tumor_GCLCposVIMpos', 'Tumor.GCLC+VIM+')
        column_name = column_name.replace('Tumor_GCLCposVIMneg', 'Tumor.GCLC+VIM-')
        column_name = column_name.replace('Tumor_GCLCnegVIMpos', 'Tumor.GCLC-VIM+')
        column_name = column_name.replace('_', '.', 1)  # Replace only first underscore
        
        # Rename the column
        temp_df.rename(columns={'Proximity Density': column_name}, inplace=True)
        
        # Merge with the result DataFrame
        if result_df.empty:
            result_df = temp_df
        else:
            result_df = pd.merge(result_df, temp_df, on='core_imageid', how='outer')


# half violin + boxplot

In [ ]:

def immune_cell_violinplots( df,
                             save_dir="violinplots_pdf",
                             palette=["#F9C3BF", "#BFE4EE", "#FBD178"],
                             palette_dot=["#fbb4b9", "#f768a1", "#d95f02"],
                             figsize=(6, 6),
                             gap=0.12,
                             box_offset=0.15):
    os.makedirs(save_dir, exist_ok=True)

    immune_cell_types = sorted(set(col.split('.')[-1] for col in df.columns if col.startswith("Tumor.GCLC")))
    group_order = ["GCLC+VIM+", "GCLC+VIM-", "GCLC-VIM+"]

    for immune_cell in immune_cell_types:
        try:
            prefix_map = {
                "GCLC+VIM+": f"Tumor.GCLC+VIM+.{immune_cell}",
                "GCLC+VIM-": f"Tumor.GCLC+VIM-.{immune_cell}",
                "GCLC-VIM+": f"Tumor.GCLC-VIM+.{immune_cell}"
            }
            selected_cols = [prefix_map[k] for k in group_order]
            if not all(col in df.columns for col in selected_cols):
                print(f"Skipping {immune_cell}, missing columns.")
                continue

            # Reshape data
            df_melt = df[selected_cols].melt(var_name="Cell_Type", value_name="Density")
            df_melt["Cell_Type"] = df_melt["Cell_Type"].map({v: k for k, v in prefix_map.items()})

            # Set up plot
            fig, ax = plt.subplots(figsize=figsize)

            # Convert group names to x-axis positions
            xpos = np.arange(len(group_order))

            # Violin plot (right half only)
            sns.violinplot(data=df_melt, x="Cell_Type", y="Density",
                           order=group_order, palette=palette, alpha=0.6,
                           cut=0, scale="width", inner=None, linewidth=0, ax=ax)

            # Clip left half of violin to simulate half violin
            for collection in ax.collections:
                path = collection.get_paths()[0]
                vertices = path.vertices
                vertices[:, 0] = np.clip(vertices[:, 0], vertices[:, 0].mean(), None)

            # Strip plot and boxplot on left side
            for i, group in enumerate(group_order):
                sub_data = df_melt[df_melt["Cell_Type"] == group]
                x_left = xpos[i] - box_offset

                # Stripplot (scatter) on left
                jittered_x = np.random.normal(x_left, 0.05, size=len(sub_data))
                ax.scatter(jittered_x,
                           sub_data["Density"],
                           color=palette_dot[i], alpha=0.5, s=15, zorder=3)
                
                # Add boxplot on left side
                boxprops = dict(facecolor=palette[i], alpha=0.7, linewidth=1.5)
                whiskerprops = dict(linewidth=1.5)
                medianprops = dict(color='black', linewidth=1.5)
                
                # Create boxplot at the left position
                ax.boxplot([sub_data["Density"]], positions=[x_left], 
                           widths=0.3, patch_artist=True,
                           boxprops=boxprops, whiskerprops=whiskerprops,
                           medianprops=medianprops, showfliers=False)
                


            # Customize axes
            ax.set_xticks(xpos)
            ax.set_xticklabels(group_order)
            ax.set_xlim(-0.5, len(group_order) - 0.5)
            ax.set_ylabel(f"Density of {immune_cell.replace('_', ' ')}")
            ax.set_xlabel("Tumor cell phenotype")
            plt.title(f"{immune_cell.replace('_', ' ')} across phenotypes")

            # Statistical annotations
            y_max = df_melt["Density"].max()
            comparisons = [(group_order[i], group_order[j]) for i in range(len(group_order)) for j in range(i + 1, len(group_order))]
            for i, (g1, g2) in enumerate(comparisons):
                v1 = df_melt[df_melt["Cell_Type"] == g1]["Density"]
                v2 = df_melt[df_melt["Cell_Type"] == g2]["Density"]
                stat, p = wilcoxon(v1, v2, alternative='two-sided')
                x1, x2 = xpos[group_order.index(g1)], xpos[group_order.index(g2)]
                y = y_max + (i + 1) * 0.05
                ax.plot([x1, x1, x2, x2], [y, y + 0.01, y + 0.01, y], lw=1.5, c='k')
                ax.text((x1 + x2) / 2, y + 0.012, f"p = {p:.2e}", ha='center', va='bottom')

            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"{immune_cell}.pdf"), dpi=300)
            plt.close()
            print(f"Saved: {immune_cell}.pdf")

        except Exception as e:
            print(f"Error processing {immune_cell}: {e}")


## Per patient

In [ ]:
pscore_data = result_df

In [ ]:
# Merge pscore_data with patient_core to add patient information
patient_core = adata.obs[['patient_id_AB19_1654', 'core_imageid']].copy()

patient_mapping = patient_core.drop_duplicates(subset=['core_imageid'])[['core_imageid', 'patient_id_AB19_1654']]

pscore_data_with_patient = pd.merge(
    pscore_data,
    patient_mapping,
    on='core_imageid',
    how='left'
)

pscore_data_patients = pscore_data_with_patient.groupby("patient_id_AB19_1654").mean(numeric_only=True)

In [ ]:
immune_cell_violinplots(pscore_data_patients,
                        save_dir="/Users/wenqchen/Desktop/Projects/Auria/Plots/violinplots/per_patient",
                        palette=["#F9C3BF", "#BFE4EE", "#FBD178"],
                        palette_dot=["#EB716B", "#79ADD6", "#EE8227"],
                        figsize=(6, 6),
                        gap=0.12,
                        box_offset=0.2)